In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Titanic-Dataset.csv')
display(df)
df.drop(columns = ['Cabin', 'Name', 'Ticket', 'PassengerId'], inplace = True)       # cabin has too many nulls
df['Age'].fillna(df['Age'].median(), inplace = True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace = True)
df['Sex'] = df['Sex'].map(({'Male' : 0, 'Female' : 1}))

df = pd.get_dummies(df, columns= ['Embarked'], drop_first = True)

# X = features (everything except what we're predicting)
# y = target (what we want to predict)
X = df.drop(columns = ['Survived'])
y = df['Survived']

X_test, X_train, y_test, y_train = train_test_split(X, y, test_size = 0.2, random_state = 42)

display('Ready')

In [ ]:
# Train the Decision Tree
# create the model: max_depth limits 
model_dt = DecisionTreeClassifier(max_depth = 4, random_state = 42)

# trains the model on training data
model_dt.fit(X_train, y_train)

# make predictions on test data 
y_pred_dt = model_dt.predict(X_test)

# evaluate
display("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
display(classification_report(y_test, y_pred_dt))

under fitting vs over fitting

- too shallow (max_depth = 2) -> underfits -> model misses patterns as it's too simple
- too deep (no limit) -> overfits -> model just memorizes the data instead of learning it, making it fail on new data
- sweet spot is in between which we will figure out now:

In [ ]:
# trying different depths to see which gives best accuracy

for depth in [2,3,4,5,6,None]:
    dt = DecisionTreeClassifier(max_depth = depth, random_state = 42)
    dt.fit(X_train, y_train)
    acc = accuracy_score(y_test, dt.predict(X_test))
    print(f"max_depth = {depth} -> Accuracy: {acc:.4f}")

now we know that max_depth = 6 is the best one
now we can proceed to visualize the tree

In [ ]:
# Train the model with the best depth
best_dt = DecisionTreeClassifier(max_depth = 6, random_state = 42)
best_dt.fit(X_train, y_train)

# visualize
plt.figure(figsize = (20,10))
plot_tree(best_dt,
        feature_names = X.columns.tolist(),
        class_names = ['Did not survive', 'Survived'],
        filled = True,      # Colour nodes by majority class
        fontsize = 8)  
plt.title('Decision Tree - Titanic Survival')
plt.tight_layout()
plt.show()